<a href="https://colab.research.google.com/github/icarobregon/AI-Engineering-2026/blob/master/sesion-01_llms-setup/src/Sesi%C3%B3n_01_Nivel_03_Openai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!pip install openai

In [29]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

from openai import OpenAI
client = OpenAI()

def calculate_api_cost(response):
    # Precios actualizados según documentación oficial (en USD por millón de tokens)
    # Incluye todos los modelos actuales (no deprecated/retired)
    # Fuente: https://developers.openai.com/api/docs/pricing?latest-pricing=standard#text-tokens
    PRICING = {
        "gpt-5.5-pro-2026-04-23": { "input": 30.00, "output": 180.00 },
        "gpt-5.4-pro-2026-03-05": { "input": 30.00, "output": 180.00 },
        "gpt-5.2-pro-2025-12-11": { "input": 21.00, "output": 168.00 },
        "gpt-5-pro-2025-10-06": { "input": 15.00, "output": 120.00 },
        "gpt-5.5-2026-04-23": { "input": 5.00, "output": 30.00 },
        "gpt-5.4-2026-03-05": { "input": 2.50, "output": 15.00 },
        "gpt-5-2025-08-07": { "input": 1.25, "output": 10.00 },
        "gpt-5.4-mini-2026-03-17": { "input": 0.75, "output": 4.50 },
        "gpt-5.4-nano-2026-03-17": { "input": 0.20, "output": 1.25 },
        "gpt-5.2-2025-12-11": { "input": 1.75, "output": 14.00 },
        "gpt-5.1-2025-11-13": { "input": 1.25, "output": 10.00 },
        "gpt-5-mini-2025-08-07": { "input": 0.25, "output": 2.00 },
        "gpt-5-nano-2025-08-07": { "input": 0.05, "output": 0.40 },
        "gpt-5-chat-latest-2025-08-07": { "input": 5.00, "output": 30.00 },
        "gpt-4.1-2025-04-14": { "input": 2.00, "output": 8.00 },
        "gpt-4.1-mini-2025-04-14": { "input": 0.40, "output": 1.60 },
        "gpt-4.1-nano-2025-04-14": { "input": 0.10, "output": 0.40 },
        "o3-2025-04-16": { "input": 2.00, "output": 8.00 },
        "o4-mini-2025-04-16": { "input": 1.10, "output": 4.40 },
        "o1-pro": { "input": 150.00, "output": 600.00 },
        "o1-pro-2025-03-19": { "input": 150.00, "output": 600.00 },
        "o3-mini-2025-01-31": { "input": 1.10, "output": 4.40 },
        "o1-2024-12-17": { "input": 15.00, "output": 60.00 },
        "o1-mini-2024-09-12": { "input": 1.10, "output": 4.40 },
        "o1-preview": { "input": 15.00, "output": 60.00 },
        "gpt-4o-2024-11-20": { "input": 2.50, "output": 10.00 },
        "gpt-4o-2024-08-06": { "input": 2.50, "output": 10.00 },
        "gpt-4o-mini-2024-07-18": { "input": 0.15, "output": 0.60 },
        "gpt-4-turbo-2024-04-09": { "input": 10.00, "output": 30.00 },
        "gpt-4-0613": { "input": 30.00, "output": 60.00 },
        "gpt-3.5-turbo-0125": { "input": 0.50, "output": 1.50 },
    }

    model = response.model
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens

    # Validar que el modelo existe en nuestro diccionario
    if model not in PRICING:
        available_models = ", ".join(sorted(set(PRICING.keys())))
        raise ValueError(
            f"❌ Modelo '{model}' no encontrado en tabla de precios.\n"
            f"Modelos disponibles:\n{available_models}\n\n"
            f"ℹ️  Nota: Los modelos deprecated o retired no están soportados.\n"
            f"Para más info: https://platform.claude.com/docs/en/about-claude/model-deprecations"
        )

    # Obtener precios del modelo
    prices = PRICING[model]

    # Calcular costes (precios están en USD por 1M tokens)
    input_cost = (input_tokens / 1_000_000) * prices["input"]
    output_cost = (output_tokens / 1_000_000) * prices["output"]
    total_cost = input_cost + output_cost

    return {
        "model": model,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
        "total_cost_formatted": f"${total_cost:.6f}"
    }

def print_cost_analysis(cost_info):
    print(f"\n💰 Análisis de Costes:")
    print(f"   Modelo:         {cost_info['model']}")
    print(f"   Input tokens:   {cost_info['input_tokens']:,}")
    print(f"   Output tokens:  {cost_info['output_tokens']:,}")
    print(f"   Total tokens:   {cost_info['input_tokens'] + cost_info['output_tokens']:,}")
    print(f"   ─────────────────────────────")
    print(f"   Coste input:    ${cost_info['input_cost']:.8f}")
    print(f"   Coste output:   ${cost_info['output_cost']:.8f}")
    print(f"   💵 Total:        {cost_info['total_cost_formatted']}")

def main():
    response = client.responses.create(
        model="gpt-5.4-nano",
        input="Actua como un Devops Senior de una empresa de desarrollo de software. Quiero alojar un proyecto personal desarrollado en Next.js y que necesita una base de datos PostgreSQL. Recomiéndame como hacerlo."
    )

    cost_info = calculate_api_cost(response)
    print_cost_analysis(cost_info)

if __name__ == "__main__":
    main()



💰 Análisis de Costes:
   Modelo:         gpt-5.4-nano-2026-03-17
   Input tokens:   48
   Output tokens:  1,141
   Total tokens:   1,189
   ─────────────────────────────
   Coste input:    $0.00000960
   Coste output:   $0.00142625
   💵 Total:        $0.001436
